# Stage2 Integrated Gradients by Scale

This notebook computes the attribution relationship between macro variables and micro variables for every retained scale under `loc_model_stage2/stage2_macro`.

For each scale, it:

1. reloads the trained stage-2 checkpoint,
2. uses the encoder output at that scale as the attribution target,
3. computes Integrated Gradients from micro inputs to each macro variable,
4. saves the attribution matrices and performance metrics,
5. draws a heatmap overview across scales.


In [1]:
from __future__ import annotations

import math
import re
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src" / "models_macro.py").exists() and (candidate / "src" / "models_micro.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from src.ig_notebook_support import (
    CompatibleMacroParellelRenormDynamic,
    load_ig_origin_data,
    resolve_preferred_device,
)
from src.models_macro import _build_windows_from_series as build_macro_windows_from_series
from src.models_micro import Parellel_Renorm_Dynamic as MicroParellelRenormDynamic
from src.models_micro import _build_windows_from_series as build_micro_windows_from_series

DEFAULT_RUN_NAME = "stage2_macro"
DEFAULT_DATA_PATH = Path("loc_data_kuramoto") / "generated_data.npz"
DEFAULT_DATA_ARRAY_KEY = "data"
DEFAULT_MAX_INITIAL_POINTS = 5
DEFAULT_DEVICE = resolve_preferred_device("cuda:6")

try:
    from captum.attr import IntegratedGradients  # type: ignore
except ModuleNotFoundError:
    class IntegratedGradients:
        """A lightweight fallback with the Captum call pattern used in this repo."""

        def __init__(self, forward_func):
            self.forward_func = forward_func

        def attribute(
            self,
            inputs: torch.Tensor,
            baselines: Optional[torch.Tensor] = None,
            target: Optional[int] = None,
            n_steps: int = 32,
            method: str = "gausslegendre",
            return_convergence_delta: bool = False,
        ):
            device = inputs.device
            dtype = inputs.dtype
            inputs_detached = inputs.detach()
            baselines = torch.zeros_like(inputs_detached) if baselines is None else baselines.detach()

            if method == "gausslegendre":
                alpha_np, weight_np = np.polynomial.legendre.leggauss(int(n_steps))
                alphas = torch.tensor((alpha_np + 1.0) / 2.0, device=device, dtype=dtype)
                weights = torch.tensor(weight_np / 2.0, device=device, dtype=dtype)
            else:
                alphas = torch.linspace(0.0, 1.0, int(n_steps), device=device, dtype=dtype)
                weights = torch.full_like(alphas, 1.0 / max(int(n_steps), 1))

            total_gradients = torch.zeros_like(inputs_detached)
            for alpha, weight in zip(alphas, weights):
                scaled_inputs = (baselines + alpha * (inputs_detached - baselines)).detach().requires_grad_(True)
                outputs = self.forward_func(scaled_inputs)
                if target is None:
                    scalar_output = outputs.reshape(outputs.shape[0], -1).sum(dim=1).sum()
                else:
                    scalar_output = outputs[:, int(target)].sum()
                gradients = torch.autograd.grad(scalar_output, scaled_inputs)[0]
                total_gradients = total_gradients + gradients * weight

            attributions = (inputs_detached - baselines) * total_gradients
            if not return_convergence_delta:
                return attributions

            with torch.no_grad():
                outputs_input = self.forward_func(inputs_detached)
                outputs_baseline = self.forward_func(baselines)
                if target is None:
                    true_diff = (outputs_input - outputs_baseline).reshape(outputs_input.shape[0], -1).sum(dim=1)
                else:
                    true_diff = outputs_input[:, int(target)] - outputs_baseline[:, int(target)]
                approximation = attributions.reshape(attributions.shape[0], -1).sum(dim=1)
                delta = true_diff - approximation
            return attributions, delta

In [2]:
def find_project_root(start: Optional[Path] = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "models_macro.py").exists() and (candidate / "src" / "models_micro.py").exists() and (candidate / "loc_model_stage2").exists():
            return candidate
    raise FileNotFoundError("Could not locate the causal_network_mix_2_0.2_syn2 project root.")


def parse_reduce_dims(value) -> List[int]:
    if value is None:
        return []
    if isinstance(value, float) and math.isnan(value):
        return []
    text = str(value).strip()
    if text == "" or text.lower() == "nan":
        return []
    return [int(part.strip()) for part in text.split(",") if part.strip()]


def extract_index(path: Path, pattern: str) -> Optional[int]:
    match = re.search(pattern, path.name)
    return None if match is None else int(match.group(1))


def list_stage2_scale_artifacts(project_root: Path, run_name: str = DEFAULT_RUN_NAME) -> List[Dict]:
    model_dir = project_root / "loc_model_stage2" / run_name
    result_dir = project_root / "loc_result_stage2" / run_name
    if not model_dir.exists():
        raise FileNotFoundError(f"Missing model directory: {model_dir}")
    if not result_dir.exists():
        raise FileNotFoundError(f"Missing result directory: {result_dir}")

    model_map = {
        idx: path
        for path in sorted(model_dir.glob("model_scale*.pkl"))
        for idx in [extract_index(path, r"model_scale(\d+)\.pkl")]
        if idx is not None
    }
    summary_map = {
        idx: path
        for path in sorted(result_dir.glob("summary_scale*.csv"))
        for idx in [extract_index(path, r"summary_scale(\d+)\.csv")]
        if idx is not None
    }

    artifacts = []
    for file_scale in sorted(set(model_map) | set(summary_map)):
        model_path = model_map.get(file_scale)
        summary_path = summary_map.get(file_scale)
        if model_path is None or summary_path is None:
            continue
        
        summary_row = pd.read_csv(summary_path).iloc[0].to_dict()
        scale_dims = parse_reduce_dims(summary_row.get("scale_dims", ""))
        reduce_dims = parse_reduce_dims(summary_row.get("reduce_dims", ""))
        logical_scale_id = int(summary_row.get("scale_id", file_scale))
        scale_dim = None
        if scale_dims and 0 <= logical_scale_id < len(scale_dims):
            scale_dim = int(scale_dims[logical_scale_id])
        elif reduce_dims:
            if file_scale == 0 and len(reduce_dims) == 1:
                scale_dim = int(reduce_dims[0])
            elif 0 <= logical_scale_id < len(reduce_dims):
                scale_dim = int(reduce_dims[logical_scale_id])

        is_micro_scale = summary_path.stem == "summary_scale0" and model_path.stem == "model_scale0"

        artifacts.append(
            {
                "file_scale": int(file_scale),
                "model_family": "micro" if is_micro_scale else "macro",
                "logical_scale_id": logical_scale_id,
                "scale_dim": scale_dim,
                "hidden_units1": int(summary_row.get("hidden_units1", 100)),
                "hidden_units2": int(summary_row.get("hidden_units2", 100)),
                "flow_num_layers": int(summary_row.get("flow_num_layers", 3)),
                "dynamics_num_layers": int(summary_row.get("dynamics_num_layers", 4)),
                "latent_size": int(summary_row.get("latent_size", 1)),
                "time_delay": int(1),
                "batch_size": int(summary_row.get("batch_size", 1024)),
                "reduce_dims": reduce_dims,
                "val_mse": float(summary_row.get("val_mse", float("nan"))),
                "test_mse": float(summary_row.get("test_mse", float("nan"))),
                "summary_path": summary_path,
                "model_path": model_path,
            }
        )

    if not artifacts:
        raise FileNotFoundError("No paired stage2 macro artifacts were found.")
    return artifacts


def load_stage2_origin_data(
    project_root: Path,
    data_path: Optional[Path] = None,
    data_key: str = DEFAULT_DATA_ARRAY_KEY,
    max_initial_points: Optional[int] = DEFAULT_MAX_INITIAL_POINTS,
) -> np.ndarray:
    resolved_path = project_root / (data_path or DEFAULT_DATA_PATH)
    return load_ig_origin_data(
        resolved_path,
        data_key=data_key,
        max_initial_points=max_initial_points,
    )


def infer_expected_micro_dim(model_path: Union[str, Path]) -> int:
    state_dict = torch.load(Path(model_path), map_location="cpu")
    candidate_dims = [
        int(value.shape[1])
        for key, value in state_dict.items()
        if getattr(value, "ndim", 0) == 2 and "flow.encoder.0.weight" in key
    ]
    if not candidate_dims:
        candidate_dims = [int(value.shape[1]) for value in state_dict.values() if getattr(value, "ndim", 0) == 2]
    if not candidate_dims:
        raise ValueError(f"Could not infer the expected micro dimension from checkpoint: {model_path}")
    return int(max(candidate_dims))


def build_fallback_micro_inputs(num_samples: int, input_dim: int, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(int(seed))
    return rng.normal(loc=0.0, scale=1.0, size=(int(num_samples), int(input_dim))).astype(np.float32)


def prepare_scale_micro_inputs(
    artifact: Dict,
    origin_data: Optional[np.ndarray],
    max_samples: Optional[int] = None,
    use_train_split: bool = True,
) -> Dict[str, Union[np.ndarray, Optional[np.ndarray], int, float, str, bool]]:
    expected_input_dim = infer_expected_micro_dim(artifact["model_path"])
    num_samples = int(max_samples) if max_samples is not None else min(int(artifact.get("batch_size", 128)), 128)
    data_input_dim = float("nan") if origin_data is None else int(origin_data.shape[-1])

    if origin_data is not None and int(origin_data.shape[-1]) == expected_input_dim:
        try:
            x_np, y_np = build_state_windows(
                origin_data,
                time_delay=artifact["time_delay"],
                model_family=artifact["model_family"],
                max_samples=max_samples,
                use_train_split=use_train_split,
            )
            return {
                "current_micro_np": extract_current_inputs(x_np),
                "next_micro_np": extract_first_step_targets(y_np),
                "expected_input_dim": int(expected_input_dim),
                "data_input_dim": data_input_dim,
                "input_source": "origin_data",
                "used_ground_truth_next": True,
            }
        except ValueError:
            pass

    if origin_data is None:
        input_source = "random_normal_no_data"
    elif int(origin_data.shape[-1]) != expected_input_dim:
        input_source = f"random_normal_dim_mismatch_{int(origin_data.shape[-1])}_to_{expected_input_dim}"
    else:
        input_source = "random_normal_window_fallback"

    current_micro_np = build_fallback_micro_inputs(
        num_samples=num_samples,
        input_dim=expected_input_dim,
        seed=2026 + int(artifact["file_scale"]),
    )
    return {
        "current_micro_np": current_micro_np,
        "next_micro_np": None,
        "expected_input_dim": int(expected_input_dim),
        "data_input_dim": data_input_dim,
        "input_source": input_source,
        "used_ground_truth_next": False,
    }


def build_state_windows(
    origin_data: np.ndarray,
    time_delay: int,
    model_family: str,
    max_samples: Optional[int] = None,
    use_train_split: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    window_builder = build_micro_windows_from_series if model_family == "micro" else build_macro_windows_from_series
    x_all, y_all = window_builder(origin_data, int(time_delay))
    if use_train_split:
        train_end = max(1, int(len(x_all) * 0.95))
        x_all = x_all[:train_end]
        y_all = y_all[:train_end]
    if max_samples is not None:
        x_all = x_all[: int(max_samples)]
        y_all = y_all[: int(max_samples)]
    if len(x_all) == 0:
        raise ValueError("No windows were generated for IG attribution.")
    return x_all.astype(np.float32), y_all.astype(np.float32)


def extract_current_inputs(x_windows: np.ndarray) -> np.ndarray:
    if x_windows.ndim == 3:
        return np.asarray(x_windows[:, 0], dtype=np.float32)
    if x_windows.ndim == 2:
        return np.asarray(x_windows, dtype=np.float32)
    raise ValueError(f"x_windows must have shape [B, T, N] or [B, N], but got {x_windows.shape}")


def extract_first_step_targets(y_windows: np.ndarray) -> np.ndarray:
    if y_windows.ndim == 3:
        return np.asarray(y_windows[:, 0], dtype=np.float32)
    if y_windows.ndim == 2:
        return np.asarray(y_windows, dtype=np.float32)
    raise ValueError(f"y_windows must have shape [B, T, N] or [B, N], but got {y_windows.shape}")


def build_scale_model(num_nodes: int, artifact: Dict, device: Union[str, torch.device] = "cpu"):
    device = torch.device(device)
    model_cls = MicroParellelRenormDynamic if artifact["model_family"] == "micro" else CompatibleMacroParellelRenormDynamic
    model = model_cls(
        sym_size=int(num_nodes),
        latent_size=int(artifact["latent_size"]),
        effect_size=int(num_nodes),
        cut_size=2,
        hidden_units1=int(artifact["hidden_units1"]),
        hidden_units2=int(artifact["hidden_units2"]),
        normalized_state=False,
        device=device,
        is_random=False,
        flow_num_layers=int(artifact["flow_num_layers"]),
        dynamics_num_layers=int(artifact["dynamics_num_layers"]),
        decode_noise_scale=0.0,
        reduce_dims=artifact["reduce_dims"],
    ).to(device)
    state_dict = torch.load(artifact["model_path"], map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


In [6]:
def compute_scale_ig_result(
    artifact: Dict,
    origin_data: Optional[np.ndarray],
    device: Union[str, torch.device] = "cpu",
    max_samples: int = 128,
    n_steps: int = 32,
    use_train_split: bool = True,
) -> Dict:
    device = torch.device(device)
    input_bundle = prepare_scale_micro_inputs(
        artifact,
        origin_data=origin_data,
        max_samples=max_samples,
        use_train_split=use_train_split,
    )
    current_micro_np = np.asarray(input_bundle["current_micro_np"], dtype=np.float32)
    next_micro_np = input_bundle["next_micro_np"]
    
    current_micro = torch.tensor(current_micro_np, dtype=torch.float32, device=device)
    next_micro = None if next_micro_np is None else torch.tensor(np.asarray(next_micro_np, dtype=np.float32), dtype=torch.float32, device=device)
    baselines = torch.zeros_like(current_micro)
    
    model = build_scale_model(int(input_bundle["expected_input_dim"]), artifact, device=device)
    scale_id = int(artifact["logical_scale_id"])
    
    def encoding_function(input_tensor: torch.Tensor) -> torch.Tensor:
        return model.encoding1(input_tensor, scale_id)[scale_id]

    with torch.no_grad():
        current_macro = encoding_function(current_micro)
    scale_dim = int(current_macro.shape[1])

    ig = IntegratedGradients(encoding_function)
    abs_rows = []
    signed_rows = []
    delta_rows = []
    for macro_idx in range(scale_dim):
        attributions, delta = ig.attribute(
            current_micro,
            baselines=baselines,
            target=macro_idx,
            n_steps=n_steps,
            method="gausslegendre",
            return_convergence_delta=True,
        )
        attributions_np = attributions.detach().cpu().numpy().astype(np.float32)
        delta_np = delta.detach().cpu().numpy().astype(np.float32)
        abs_rows.append(np.mean(np.abs(attributions_np), axis=0))
        signed_rows.append(np.mean(attributions_np, axis=0))
        delta_rows.append(delta_np)

    abs_attribution_matrix = np.stack(abs_rows, axis=0).astype(np.float32)
    signed_attribution_matrix = np.stack(signed_rows, axis=0).astype(np.float32)
    delta_array = np.stack(delta_rows, axis=0).astype(np.float32)

    with torch.no_grad():
        reconstructed_micro = model.decoding(current_macro, scale_id)
        predicted_micro_list, _, predicted_macro_rollout = model.train_forward(current_micro, scale_id=scale_id, delay=1)
        predicted_next_micro = predicted_micro_list[0][:, 0]
        predicted_next_macro = predicted_macro_rollout[0][:, 0]

    reconstruction_mae = float(torch.mean(torch.abs(reconstructed_micro - current_micro)).item())
    reconstruction_mse = float(torch.mean((reconstructed_micro - current_micro) ** 2).item())

    if next_micro is not None:
        with torch.no_grad():
            next_macro = encoding_function(next_micro)
        macro_transition_mae = float(torch.mean(torch.abs(predicted_next_macro - next_macro)).item())
        macro_transition_mse = float(torch.mean((predicted_next_macro - next_macro) ** 2).item())
        decoded_next_micro_mae = float(torch.mean(torch.abs(predicted_next_micro - next_micro)).item())
        decoded_next_micro_mse = float(torch.mean((predicted_next_micro - next_micro) ** 2).item())
    else:
        macro_transition_mae = float("nan")
        macro_transition_mse = float("nan")
        decoded_next_micro_mae = float("nan")
        decoded_next_micro_mse = float("nan")

    return {
        **artifact,
        "num_samples": int(current_micro_np.shape[0]),
        "input_dim": int(current_micro_np.shape[1]),
        "expected_input_dim": int(input_bundle["expected_input_dim"]),
        "data_input_dim": input_bundle["data_input_dim"],
        "input_source": str(input_bundle["input_source"]),
        "used_ground_truth_next": bool(input_bundle["used_ground_truth_next"]),
        "scale_dim": scale_dim,
        "abs_attribution_matrix": abs_attribution_matrix,
        "signed_attribution_matrix": signed_attribution_matrix,
        "delta_array": delta_array,
        "reconstruction_mae": reconstruction_mae,
        "reconstruction_mse": reconstruction_mse,
        "macro_transition_mae": macro_transition_mae,
        "macro_transition_mse": macro_transition_mse,
        "decoded_next_micro_mae": decoded_next_micro_mae,
        "decoded_next_micro_mse": decoded_next_micro_mse,
        "mean_abs_convergence_delta": float(np.mean(np.abs(delta_array))),
    }

def compute_ig_for_all_scales(
    project_root: Optional[Union[str, Path]] = None,
    run_name: str = DEFAULT_RUN_NAME,
    data_path: Optional[Union[str, Path]] = None,
    data_key: str = DEFAULT_DATA_ARRAY_KEY,
    max_initial_points: Optional[int] = DEFAULT_MAX_INITIAL_POINTS,
    device: Union[str, torch.device] = "cpu",
    max_samples: int = 128,
    n_steps: int = 32,
    use_train_split: bool = True,
) -> List[Dict]:
    project_root = find_project_root(Path(project_root) if project_root is not None else None)
    artifacts = list_stage2_scale_artifacts(project_root, run_name=run_name)
    print(artifacts)
    try:
        origin_data = load_stage2_origin_data(
            project_root,
            Path(data_path) if data_path is not None else None,
            data_key=data_key,
            max_initial_points=max_initial_points,
        )
    except FileNotFoundError:
        origin_data = None
    
    results = []
    for artifact in artifacts:
        results.append(
            compute_scale_ig_result(
                artifact,
                origin_data,
                device=device,
                max_samples=max_samples,
                n_steps=n_steps,
                use_train_split=use_train_split,
            )
        )
    return results

def save_ig_results(results: List[Dict], output_dir: Union[str, Path]) -> pd.DataFrame:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    summary_rows = []
    for item in results:
        macro_labels = list(range(1, item["scale_dim"] + 1))
        micro_labels = list(range(1, item["input_dim"] + 1))

        abs_path = output_dir / f"ig_abs_attribution_scale{item['file_scale']}.csv"
        signed_path = output_dir / f"ig_signed_attribution_scale{item['file_scale']}.csv"
        delta_path = output_dir / f"ig_convergence_delta_scale{item['file_scale']}.csv"

        pd.DataFrame(item["abs_attribution_matrix"], index=macro_labels, columns=micro_labels).to_csv(abs_path)
        pd.DataFrame(item["signed_attribution_matrix"], index=macro_labels, columns=micro_labels).to_csv(signed_path)
        pd.DataFrame(item["delta_array"], index=macro_labels).to_csv(delta_path, index_label="macro_variable")

        summary_rows.append(
            {
                "file_scale": item["file_scale"],
                "model_family": item["model_family"],
                "logical_scale_id": item["logical_scale_id"],
                "scale_dim": item["scale_dim"],
                "input_dim": item["input_dim"],
                "expected_input_dim": item["expected_input_dim"],
                "data_input_dim": item["data_input_dim"],
                "input_source": item["input_source"],
                "used_ground_truth_next": item["used_ground_truth_next"],
                "num_samples": item["num_samples"],
                "val_mse": item["val_mse"],
                "test_mse": item["test_mse"],
                "reconstruction_mae": item["reconstruction_mae"],
                "reconstruction_mse": item["reconstruction_mse"],
                "macro_transition_mae": item["macro_transition_mae"],
                "macro_transition_mse": item["macro_transition_mse"],
                "decoded_next_micro_mae": item["decoded_next_micro_mae"],
                "decoded_next_micro_mse": item["decoded_next_micro_mse"],
                "mean_abs_convergence_delta": item["mean_abs_convergence_delta"],
                "abs_attribution_csv": str(abs_path.resolve()),
                "signed_attribution_csv": str(signed_path.resolve()),
                "convergence_delta_csv": str(delta_path.resolve()),
            }
        )

    summary_df = pd.DataFrame(summary_rows).sort_values("file_scale").reset_index(drop=True)
    summary_df.to_csv(output_dir / "ig_scale_summary.csv", index=False)
    return summary_df

def plot_ig_heatmaps(
    results: List[Dict],
    output_dir: Union[str, Path],
    use_signed: bool = False,
    dpi: int = 200,
) -> List[Path]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    saved_paths = []
    colorbar_label = "Signed Attribution" if use_signed else "Mean |Attribution|"
    cmap = "coolwarm" if use_signed else "Reds"
    prefix = "ig_signed_heatmap" if use_signed else "ig_abs_heatmap"

    for item in results:
        matrix = item["signed_attribution_matrix"] if use_signed else item["abs_attribution_matrix"]
        figure, axis = plt.subplots(figsize=(8, 5), constrained_layout=True)
        image = axis.imshow(matrix, aspect="auto", cmap=cmap)
        axis.set_title(f"Scale {item['file_scale']} | dim={item['scale_dim']}")
        axis.set_xlabel("Micro Variable")
        axis.set_ylabel("Macro Variable")
        axis.set_xticks(np.arange(item["input_dim"]))
        axis.set_xticklabels(np.arange(1, item["input_dim"] + 1))
        axis.set_yticks(np.arange(item["scale_dim"]))
        axis.set_yticklabels(np.arange(1, item["scale_dim"] + 1))
        figure.colorbar(image, ax=axis, shrink=0.9, label=colorbar_label)

        saved_path = output_dir / f"{prefix}_scale{item['file_scale']}.png"
        figure.savefig(saved_path, dpi=dpi, bbox_inches="tight")
        display(figure)
        plt.close(figure)
        saved_paths.append(saved_path)

    return saved_paths


In [8]:
PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "loc_result_stage2" / DEFAULT_RUN_NAME / "IG"

DATA_ARRAY_KEY = DEFAULT_DATA_ARRAY_KEY
MAX_INITIAL_POINTS = DEFAULT_MAX_INITIAL_POINTS
MAX_WINDOWS_PER_SCALE = 1000
N_STEPS = 24
DEVICE = DEFAULT_DEVICE

IG_RESULTS = compute_ig_for_all_scales(
    project_root=PROJECT_ROOT,
    run_name=DEFAULT_RUN_NAME,
    data_path=DEFAULT_DATA_PATH,
    data_key=DATA_ARRAY_KEY,
    max_initial_points=MAX_INITIAL_POINTS,
    device=DEVICE,
    max_samples=MAX_WINDOWS_PER_SCALE,
    n_steps=N_STEPS,
    use_train_split=True,
)

IG_SUMMARY = save_ig_results(IG_RESULTS, output_dir=OUTPUT_DIR)
IG_ABS_HEATMAP_PATHS = plot_ig_heatmaps(IG_RESULTS, output_dir=OUTPUT_DIR, use_signed=False)
IG_SIGNED_HEATMAP_PATHS = plot_ig_heatmaps(IG_RESULTS, output_dir=OUTPUT_DIR, use_signed=True)

display(IG_SUMMARY)
IG_ABS_HEATMAP_PATHS, IG_SIGNED_HEATMAP_PATHS


[{'file_scale': 1, 'model_family': 'macro', 'logical_scale_id': 0, 'scale_dim': 32, 'hidden_units1': 50, 'hidden_units2': 50, 'flow_num_layers': 3, 'dynamics_num_layers': 5, 'latent_size': 1, 'time_delay': 1, 'batch_size': 2048, 'reduce_dims': [32, 16, 8, 4, 2, 1], 'val_mse': 0.0001188683745567, 'test_mse': 0.0019591618329286, 'summary_path': PosixPath('/home/wangzhipeng/code/causal_network/causal_network/kumamoto/causal_network_mix_2_0.2_kuma_theta_mlp1/loc_result_stage2/stage2_macro/summary_scale1.csv'), 'model_path': PosixPath('/home/wangzhipeng/code/causal_network/causal_network/kumamoto/causal_network_mix_2_0.2_kuma_theta_mlp1/loc_model_stage2/stage2_macro/model_scale1.pkl')}, {'file_scale': 2, 'model_family': 'macro', 'logical_scale_id': 1, 'scale_dim': 16, 'hidden_units1': 50, 'hidden_units2': 50, 'flow_num_layers': 3, 'dynamics_num_layers': 5, 'latent_size': 1, 'time_delay': 1, 'batch_size': 2048, 'reduce_dims': [32, 16, 8, 4, 2, 1], 'val_mse': 0.0001662926952121, 'test_mse': 0

<Figure size 800x500 with 2 Axes>

<Figure size 800x500 with 2 Axes>

<Figure size 800x500 with 2 Axes>

<Figure size 800x500 with 2 Axes>

...

<Figure size 800x500 with 2 Axes>

<Figure size 800x500 with 2 Axes>

<Figure size 800x500 with 2 Axes>

,file_scale,model_family,logical_scale_id,scale_dim,input_dim,expected_input_dim,data_input_dim,input_source,used_ground_truth_next,num_samples,...,reconstruction_mae,reconstruction_mse,macro_transition_mae,macro_transition_mse,decoded_next_micro_mae,decoded_next_micro_mse,mean_abs_convergence_delta,abs_attribution_csv,signed_attribution_csv,convergence_delta_csv
0,1,macro,0,32,32,32,32,origin_data,True,1000,...,0.007513,0.000127,0.186371,0.060119,0.321595,0.104561,0.002944,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...
1,2,macro,1,16,32,32,32,origin_data,True,1000,...,0.009750,0.000152,0.094340,0.012531,0.177909,0.031832,0.001241,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...
2,3,macro,2,8,32,32,32,origin_data,True,1000,...,0.010017,0.000151,0.088326,0.010457,0.033103,0.001147,0.000721,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...
3,4,macro,3,4,32,32,32,origin_data,True,1000,...,0.040346,0.001718,0.110109,0.014822,0.006465,0.000103,0.000526,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...
4,5,macro,4,2,32,32,32,origin_data,True,1000,...,0.006695,0.000104,0.012702,0.000288,0.030235,0.000971,0.000788,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...
5,6,macro,5,1,32,32,32,origin_data,True,1000,...,0.002526,0.000073,0.010791,0.000118,0.038240,0.001509,0.000853,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...


([PosixPath('/home/wangzhipeng/code/causal_network/causal_network/kumamoto/causal_network_mix_2_0.2_kuma_theta_mlp1/loc_result_stage2/stage2_macro/IG/ig_abs_heatmap_scale1.png'),
  PosixPath('/home/wangzhipeng/code/causal_network/causal_network/kumamoto/causal_network_mix_2_0.2_kuma_theta_mlp1/loc_result_stage2/stage2_macro/IG/ig_abs_heatmap_scale2.png'),
  PosixPath('/home/wangzhipeng/code/causal_network/causal_network/kumamoto/causal_network_mix_2_0.2_kuma_theta_mlp1/loc_result_stage2/stage2_macro/IG/ig_abs_heatmap_scale3.png'),
  PosixPath('/home/wangzhipeng/code/causal_network/causal_network/kumamoto/causal_network_mix_2_0.2_kuma_theta_mlp1/loc_result_stage2/stage2_macro/IG/ig_abs_heatmap_scale4.png'),
  PosixPath('/home/wangzhipeng/code/causal_network/causal_network/kumamoto/causal_network_mix_2_0.2_kuma_theta_mlp1/loc_result_stage2/stage2_macro/IG/ig_abs_heatmap_scale5.png'),
  PosixPath('/home/wangzhipeng/code/causal_network/causal_network/kumamoto/causal_network_mix_2_0.2_kuma_